# Lab 1D1: Query Language in Python

**Time**: ~15 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will write SQL-like queries against Azure Cosmos DB using the Python `azure-cosmos` SDK. You will query for filtered results, parameterized queries, and compare point read vs query costs.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, and `python-dotenv` installed
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint

Run each cell in order to complete the steps.

## Step 0: Initialize Connection

Set up the Cosmos client connection to the `WorkshopData/Catalog` container.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
DB_NAME = "WorkshopData"
CONT_NAME = "Catalog"

if not ENDPOINT:
    raise RuntimeError("COSMOS_ENDPOINT environment variable is required.")

print(f"  endpoint: {ENDPOINT}")
print(f"  database: {DB_NAME}")
print(f"  container: {CONT_NAME}")
print(f"  connected: {ENDPOINT}{DB_NAME}/{CONT_NAME}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import AzureCliCredential

cred = AzureCliCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)
db = client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

## Step 1: Seed Data (Prebuilt)

Seeds 5 fruit/vegetable items with grocery partition key. Each item has a `tags` array and nested `nutrition` object so later steps can demonstrate JSON queries.

In [ ]:
seed_items = [
    {
        "id": "1", "name": "Apples", "category": "fruit", "price": 1.20, "partitionKey": "grocery",
        "tags": ["organic", "seasonal", "domestic"],
        "nutrition": {"calories": 95, "vitamins": ["A", "C"]},
    },
    {
        "id": "2", "name": "Broccoli", "category": "vegetable", "price": 2.50, "partitionKey": "grocery",
        "tags": ["organic", "fresh"],
        "nutrition": {"calories": 55, "vitamins": ["C", "K", "A"]},
    },
    {
        "id": "3", "name": "Bananas", "category": "fruit", "price": 0.80, "partitionKey": "grocery",
        "tags": ["imported", "ripe", "organic"],
        "nutrition": {"calories": 105, "vitamins": ["B6", "C"]},
    },
    {
        "id": "4", "name": "Carrots", "category": "vegetable", "price": 1.00, "partitionKey": "grocery",
        "tags": ["organic", "root", "fresh"],
        "nutrition": {"calories": 41, "vitamins": ["A", "K"]},
    },
    {
        "id": "5", "name": "Dates", "category": "fruit", "price": 4.00, "partitionKey": "grocery",
        "tags": ["imported", "dried", "premium"],
        "nutrition": {"calories": 280, "vitamins": ["B6", "K"]},
    },
]

for item in seed_items:
    try:
        response = container.upsert_item(body=item)
        print(f"  Upserted: {item['name']}")
    except Exception as ex:
        print(f"  Error: {ex}")

print(f"\nSeeded {len(seed_items)} items")

## Step 2: Query for All Fruits

Write a parameterized query to retrieve items where `c.category` matches the `@cat` parameter. Replace the placeholder `query` string in the code cell with:

```python
query = "SELECT * FROM c WHERE c.category = @cat"
```

**Expected output**: 3 items (Apples, Bananas, Dates) with their prices, then total RU charged.

In [ ]:
category_to_query = "fruit"

query = "SELECT * FROM c WHERE c.category = @cat"

fruits = list(container.query_items(
    query=query,
    parameters=[{"name": "@cat", "value": category_to_query}],
    enable_cross_partition_query=True
))

fruit_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print(f"Found {len(fruits)} fruit items:")
for f in fruits:
    print(f"  {f['name']}: ${f['price']}")
print(f"RU charged: {fruit_query_ru}")

## Step 3: Point Read vs Query Cost (Prebuilt)

Fetch the same single item (`id = "1"`) two ways — a point read and a parameterized, partition-scoped `SELECT * FROM c WHERE c.id = @id` query (scoped to the `grocery` partition) — and compare their RU charges. Both target the same item, so the takeaway is that the point read is cheaper than a query for fetching one item by id.

In [ ]:
point_read_item = container.read_item(item="1", partition_key="grocery")
point_read_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

single_item_query = "SELECT * FROM c WHERE c.id = @id"
list(container.query_items(
    query=single_item_query,
    parameters=[{"name": "@id", "value": "1"}],
    partition_key="grocery"
))
single_item_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Fetching the same single item (id='1') two different ways.\n")
print(f"Point read (1 item by id + partition key): {point_read_ru} RU")
print(f"Query  (SELECT * FROM c WHERE c.id='1'):   {single_item_query_ru} RU")
if point_read_ru > 0:
    print(f"Point read is {single_item_query_ru / point_read_ru:.1f}x cheaper for fetching a single item by id.")

## Step 4: Parameterized Query

Replace the placeholder `top_query` in the code cell with a `SELECT TOP @limit` query ordered by price descending:

```python
top_query = "SELECT TOP @limit c.name, c.price FROM c ORDER BY c.price DESC"
```

**Expected output**: 3 items sorted by price descending.

In [ ]:
limit_val = 3

top_query = "SELECT TOP @limit c.name, c.price FROM c ORDER BY c.price DESC"

top_items = list(container.query_items(
    query=top_query,
    parameters=[{"name": "@limit", "value": limit_val}],
    enable_cross_partition_query=True
))

print(f"Top {limit_val} items by price:")
for item in top_items:
    print(f"  {item['name']}: ${item['price']}")

## Step 5: JSON Properties + System Functions

Cosmos DB stores items as JSON, so queries can reach into nested objects and arrays directly. Replace the placeholder `json_query` string in the code cell with:

```python
json_query = (
    "SELECT c.name, CONCAT(c.category, ' category') AS category, c.nutrition.calories "
    "FROM c "
    "WHERE ARRAY_CONTAINS(c.tags, 'organic') AND c.nutrition.calories < 100"
)
```

This combines nested-property access (`c.nutrition.calories`), the `ARRAY_CONTAINS` system function on the `tags` array, and `CONCAT` in the projection. See the [system functions reference](https://learn.microsoft.com/azure/cosmos-db/nosql/query/system-functions) for the full list.

**Expected output**: Apples, Broccoli, and Carrots — the organic items under 100 calories. (Organic Bananas at 105 calories are excluded by the < 100 filter.)

In [ ]:
json_query = (
    "SELECT c.name, CONCAT(c.category, ' category') AS category, c.nutrition.calories "
    "FROM c "
    "WHERE ARRAY_CONTAINS(c.tags, 'organic') AND c.nutrition.calories < 100"
)

results = list(container.query_items(
    query=json_query,
    enable_cross_partition_query=True
))

json_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Low-calorie organic items:")
for item in results:
    print(f"  {item['name']} ({item['category']}): {item['calories']} cal")
print(f"RU charged: {json_query_ru}")

## Step 6: Subquery Over a Nested Array

A [subquery](https://learn.microsoft.com/azure/cosmos-db/nosql/query/subquery) lets a query iterate or aggregate over a nested array inside each document. Replace the placeholder `subquery` string in the code cell with:

```python
subquery = (
    "SELECT c.name, "
    "       (SELECT VALUE COUNT(1) FROM v IN c.nutrition.vitamins) AS vitaminCount "
    "FROM c "
    "ORDER BY c.name"
)
```

**Expected output**: each item listed with its count of vitamins from `nutrition.vitamins`.

In [ ]:
subquery = (
    "SELECT c.name, "
    "       (SELECT VALUE COUNT(1) FROM v IN c.nutrition.vitamins) AS vitaminCount "
    "FROM c "
    "ORDER BY c.name"
)

results = list(container.query_items(
    query=subquery,
    enable_cross_partition_query=True
))

subquery_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Vitamin counts per item:")
for item in results:
    print(f"  {item['name']}: {item['vitaminCount']} vitamins")
print(f"RU charged: {subquery_ru}")

## Lab Complete!

You have completed the query language exercise in Python. You:
- Connected to Cosmos DB using `AzureCliCredential`
- Seeded sample data with nested objects and arrays
- Ran a filter query using `query_items()` with parameterized input
- Compared point read vs query cost
- Wrote a parameterized query with `TOP`
- Queried nested JSON properties with `ARRAY_CONTAINS` and `CONCAT`
- Wrote a subquery that aggregates a nested array

To run the lab again from scratch, re-run each cell in order.